In [4]:
typealias Name = String
data class TransferDetails(
    val fromPurse: Name,
    val toPurse: Name,
    val value: ULong
)
data class PayDetails(
    val td: TransferDetails,
    val fromSeqNo: ULong,
    val toSeqNo: ULong,
)
fun allSimpleULongs(): Sequence<ULong> = generateSequence(0uL) { it + 1uL }
fun allSimpleNames(): Sequence<Name> = allSimpleULongs().map { "name$it" }
fun allPayDetails(
    // This is not quite like the Z (e.g. generation is deterministic and progressive; in Z its non-deterministic)
    fromNames: Sequence<Name> = allSimpleNames(),
    toNames: Sequence<Name> = allSimpleNames(),
    values: Sequence<ULong> = generateSequence(0uL) { it + 1uL },
    fromSeqNos: Sequence<ULong> = generateSequence(0uL) { it + 1uL },
    toSeqNos: Sequence<ULong> = generateSequence(0uL) { it + 1uL },
): Sequence<PayDetails> {
    return fromNames.flatMap { from ->
        toNames.flatMap { to ->
            if (from == to) emptySequence()
            else values.flatMap { value ->
                fromSeqNos.flatMap { fromSeq ->
                    toSeqNos.map { toSeq ->
                        PayDetails(
                            td = TransferDetails(from, to, value),
                            fromSeqNo = fromSeq,
                            toSeqNo = toSeq,
                        )
                    }
                }
            }
        }
    }
}

val abWorld = mapOf("leo" to PayDetails(TransferDetails("leo", "erin", 10U), 0U, 0U),
    "erin" to PayDetails(TransferDetails("erin", "leo", 10U), 0U, 0U))
val from = allPayDetails(fromNames = abWorld.keys.asSequence())
println(from.take(10).toList())

/** O(n) element access — fine for exploration; cache if you need speed. */
fun <T> Sequence<T>.nth(n: Int): T = drop(n).first()

/** All index tuples of length [dims] whose components sum to [sum]. */
fun tuplesWithSum(dims: Int, sum: Int): Sequence<List<Int>> = sequence {
    if (dims == 1) {
        yield(listOf(sum))
        return@sequence
    }
    for (head in 0..sum) {
        tuplesWithSum(dims - 1, sum - head).forEach { tail ->
            yield(listOf(head) + tail)
        }
    }
}
fun allPayDetailsFair(
    fromNames: Sequence<Name> = allSimpleNames(),
    toNames: Sequence<Name> = allSimpleNames(),
    values: Sequence<ULong> = allSimpleULongs(),
    fromSeqNos: Sequence<ULong> = allSimpleULongs(),
    toSeqNos: Sequence<ULong> = allSimpleULongs(),
): Sequence<PayDetails> = sequence {
    val froms = fromNames.toList()   // finite in your ConWorld case
    val seqs = listOf(toNames, values, fromSeqNos, toSeqNos)

    var sum = 0
    while (true) {
        for (fi in froms.indices) {
            tuplesWithSum(4, sum).forEach { (ti, vi, fsi, tsi) ->
                val from = froms[fi]
                val to = toNames.nth(ti)
                if (from != to) {
                    yield(
                        PayDetails(
                            td = TransferDetails(from, to, values.nth(vi)),
                            fromSeqNo = fromSeqNos.nth(fsi),
                            toSeqNo = toSeqNos.nth(tsi),
                        )
                    )
                }
            }
        }
        sum++
    }
}
val from2 = allPayDetailsFair(fromNames = abWorld.keys.asSequence())
//println(from2.take(100).toList())

[PayDetails(td=TransferDetails(fromPurse=leo, toPurse=name0, value=0), fromSeqNo=0, toSeqNo=0), PayDetails(td=TransferDetails(fromPurse=leo, toPurse=name0, value=0), fromSeqNo=0, toSeqNo=1), PayDetails(td=TransferDetails(fromPurse=leo, toPurse=name0, value=0), fromSeqNo=0, toSeqNo=2), PayDetails(td=TransferDetails(fromPurse=leo, toPurse=name0, value=0), fromSeqNo=0, toSeqNo=3), PayDetails(td=TransferDetails(fromPurse=leo, toPurse=name0, value=0), fromSeqNo=0, toSeqNo=4), PayDetails(td=TransferDetails(fromPurse=leo, toPurse=name0, value=0), fromSeqNo=0, toSeqNo=5), PayDetails(td=TransferDetails(fromPurse=leo, toPurse=name0, value=0), fromSeqNo=0, toSeqNo=6), PayDetails(td=TransferDetails(fromPurse=leo, toPurse=name0, value=0), fromSeqNo=0, toSeqNo=7), PayDetails(td=TransferDetails(fromPurse=leo, toPurse=name0, value=0), fromSeqNo=0, toSeqNo=8), PayDetails(td=TransferDetails(fromPurse=leo, toPurse=name0, value=0), fromSeqNo=0, toSeqNo=9)]
